# Cắt clip (bước 03) theo từng part trên Kaggle
Thay cho `data_pipeline/steps/03_cut.py` khi máy local chậm. Dùng **đúng code và `configs/data.yaml`** của local: Silero VAD + YuNet, clip 5–8 s, 25 fps CFR, cạnh ngắn ≤1080.
Đầu vào: thư mục raw của part (kết quả bước 02) đã upload thành Kaggle dataset, gồm `videos/`, `sources.jsonl`, `logs/download_results.csv`.
Bật Internet (cài thư viện, tải YuNet/Silero). Không cần GPU: VAD và dò mặt chạy CPU.
Output: `clips/` + `review.csv` + `logs/`, nén ZIP để tải về local rồi chạy `04_review.py`.

In [ ]:
from pathlib import Path
import subprocess, sys, os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-forensics.git"  # Sửa nếu đổi repository.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
PIPELINE = ROOT / "data_pipeline"
assert (PIPELINE / "src/vn_av_data/data/curation.py").is_file(), "Cần push code mới trước khi dùng notebook"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PIPELINE)], check=True)
sys.path.insert(0, str(PIPELINE / "src"))
os.chdir(PIPELINE)  # Đường dẫn model trong configs/data.yaml tính từ data_pipeline/.
from vn_av_data.cli import main

## 1. Chọn part và đường dẫn
`RAW_DATASET` là thư mục chứa `sources.jsonl` (đường dẫn video trong manifest tính tương đối từ file này, nên giữ nguyên cấu trúc khi upload).
Kaggle giới hạn thời gian phiên; cắt ghi nhật ký theo từng video nên chạy lại là cắt tiếp. Sang phiên mới: attach output (hoặc ZIP) của phiên trước rồi đặt `PREVIOUS_CUT`.
Luật cắt, code cắt và model bị khóa bằng chữ ký trong `logs/cut-run.json`: phiên sau và local phải dùng cùng code/config thì mới cắt tiếp được.

In [ ]:
PART = 1  # Ví dụ: 2 -> xử lý vn-av-df-data-part2.
DATASET_VERSION = f"vn-av-df-data-part{PART}"
RAW_DATASET = Path("/kaggle/input/YOUR_RAW_PART") / DATASET_VERSION  # Thư mục có sources.jsonl.
OUTPUT = Path("/kaggle/working/candidates") / DATASET_VERSION
PREVIOUS_CUT = None  # Cắt tiếp phiên trước, ví dụ Path("/kaggle/input/YOUR_CUT_PART") / DATASET_VERSION.
CONFIG = PIPELINE / "configs/data.yaml"  # Giữ như local để clip cùng luật.

manifest = RAW_DATASET / "sources.jsonl"
assert manifest.is_file(), f"Không thấy {manifest}; kiểm tra RAW_DATASET"
if PREVIOUS_CUT is not None and not OUTPUT.exists():
    import shutil
    shutil.copytree(PREVIOUS_CUT, OUTPUT)
    print("Cắt tiếp từ:", PREVIOUS_CUT)
print("Part:", DATASET_VERSION, "| Input:", manifest, "| Output:", OUTPUT)
print(CONFIG.read_text(encoding="utf-8"))

## 2. Tải model và cắt
Lỗi ở một video không dừng cả part: video lỗi ghi vào `logs/errors.json`, cuối lượt báo lỗi; chạy lại cell để thử lại các video đó.
Dù cell cắt báo lỗi, vẫn chạy cell ZIP để giữ phần đã cắt.

In [ ]:
main(["setup", "--config", str(CONFIG)])

In [ ]:
main(["cut", "--config", str(CONFIG), "--manifest", str(manifest), "--output", str(OUTPUT)])

In [ ]:
import json, shutil
logs = OUTPUT / "logs"
for name in ("summary.json", "errors.json"):
    if (logs / name).exists():
        print(name, json.dumps(json.loads((logs / name).read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
archive = shutil.make_archive("/kaggle/working/" + OUTPUT.name, "zip", OUTPUT.parent, OUTPUT.name)
print("Tải part:", archive)

## 3. Review ở local
Giải nén ZIP vào `data_pipeline/data/candidates/vn-av-df-data/`, được thư mục `vn-av-df-data-partN` (có `clips/`, `review.csv`, `logs/`).
Đặt `PART=N` trong `data_settings.py` rồi chạy `data_pipeline/steps/04_review.py` → `05_export.py`.
Tải thêm video cho part: có thể cắt phần mới ở local bằng `03_cut.py` (cùng code/config) hoặc upload lại raw và đặt `PREVIOUS_CUT` trên Kaggle.